# Lab 1: Kafka in Python – Producers, Consumers, and Real-Time ETL

**Business case:** Transaction monitoring system for an e-commerce company.

## Part 1: Create a Kafka Topic

Run in a JupyterLab terminal:

```bash
kafka/bin/kafka-topics.sh --create \
  --topic transactions \
  --bootstrap-server broker:9092 \
  --partitions 3 \
  --replication-factor 1

kafka/bin/kafka-topics.sh --list --bootstrap-server broker:9092
```

```
# --list output:
# transactions
```

## Part 2: Producer – Generating Transactions

### Task 2.1 – Transaction producer

In [ ]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def generate_transaction(tx_num):
    return {
        "tx_id": f"TX{tx_num:04d}",
        "user_id": f"u{random.randint(1, 20):02d}",
        "amount": round(random.uniform(5.0, 5000.0), 2),
        "store": random.choice(["Warsaw", "Krakow", "Gdansk", "Wroclaw"]),
        "category": random.choice(["electronics", "clothing", "food", "books"]),
        "timestamp": datetime.now().isoformat()
    }

print("Starting producer – sending 50 transactions to 'transactions' topic...")

for i in range(1, 51):
    tx = generate_transaction(i)
    producer.send('transactions', value=tx)
    print(f"Sent: {tx['tx_id']} | {tx['user_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']}")
    time.sleep(1)

producer.flush()
print("Done – 50 transactions sent.")

Run in terminal: `python producer.py`

## Part 3: Stateless Consumer – Filtering

### Task 3.1 – Filter large transactions

In [ ]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='filter-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Listening for large transactions (amount > 1000)...")

for msg in consumer:
    tx = msg.value
    if tx['amount'] > 1000:
        print(f"ALERT: {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']}")

### Task 3.2 – Transform and enrich

In [ ]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='enrich-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Enriching transactions with risk_level...")

for msg in consumer:
    tx = msg.value
    amount = tx['amount']

    if amount > 3000:
        tx['risk_level'] = 'HIGH'
    elif amount > 1000:
        tx['risk_level'] = 'MEDIUM'
    else:
        tx['risk_level'] = 'LOW'

    print(f"[{tx['risk_level']:6s}] {tx['tx_id']} | {amount:.2f} PLN | {tx['store']} | {tx['category']}")

## Part 4: Stateful Consumer – Aggregating

### Task 4.1 – Count transactions per store

In [ ]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

print("Counting transactions per store (summary every 10 messages)...")

for msg in consumer:
    tx = msg.value
    store = tx['store']
    amount = tx['amount']

    store_counts[store] += 1
    total_amount[store] = total_amount.get(store, 0.0) + amount
    msg_count += 1

    if msg_count % 10 == 0:
        print(f"\n--- Summary after {msg_count} messages ---")
        print(f"{'Store':<12} {'Count':>6} {'Total Amount':>14} {'Avg Amount':>12}")
        print("-" * 48)
        for s in sorted(store_counts):
            count = store_counts[s]
            total = total_amount[s]
            avg = total / count
            print(f"{s:<12} {count:>6} {total:>14.2f} {avg:>12.2f}")

### Task 4.2 – Running statistics per category

In [ ]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='stats-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# Per-category state: {category: {count, total, min, max}}
stats = defaultdict(lambda: {'count': 0, 'total': 0.0, 'min': float('inf'), 'max': float('-inf')})
msg_count = 0

print("Tracking per-category statistics (summary every 10 messages)...")

for msg in consumer:
    tx = msg.value
    cat = tx['category']
    amount = tx['amount']

    s = stats[cat]
    s['count'] += 1
    s['total'] += amount
    s['min'] = min(s['min'], amount)
    s['max'] = max(s['max'], amount)
    msg_count += 1

    if msg_count % 10 == 0:
        print(f"\n--- Category Stats after {msg_count} messages ---")
        print(f"{'Category':<14} {'Count':>6} {'Total':>12} {'Min':>9} {'Max':>9} {'Avg':>9}")
        print("-" * 63)
        for cat_name in sorted(stats):
            s = stats[cat_name]
            avg = s['total'] / s['count']
            print(f"{cat_name:<14} {s['count']:>6} {s['total']:>12.2f} {s['min']:>9.2f} {s['max']:>9.2f} {avg:>9.2f}")

## Part 5: Multiple Consumers

### Task 5.1 – Run everything together

Open 3 terminals in JupyterLab and run simultaneously:
1. `python producer.py`
2. `python consumer_filter.py`
3. `python consumer_count.py`

### Task 5.2 – Questions

In [ ]:
# Answer these questions:
#
# 1. What happens if you start consumer_filter.py AFTER the producer has finished?
#    Because auto_offset_reset='earliest', the consumer will start reading from the
#    BEGINNING of the topic and process all 50 already-sent messages. No messages are lost.
#
# 2. What happens if two consumers have the SAME group_id?
#    Kafka treats them as a single consumer group and distributes the partitions between them.
#    With 3 partitions and 2 consumers, one consumer gets 2 partitions and the other gets 1.
#    Each message is delivered to only ONE of the two consumers (load balancing, not broadcast).
#
# 3. What is the difference between stateless and stateful processing?
#    Stateless: each event is processed independently with no memory of past events.
#      Example from this lab: consumer_filter.py – each transaction is evaluated on its own
#      (amount > 1000) without knowing anything about previous transactions.
#    Stateful: the consumer maintains state (memory) across events.
#      Example from this lab: consumer_count.py – store_counts and total_amount accumulate
#      across all messages; the summary depends on all previously seen transactions.

## Homework: Velocity Anomaly Detection

In [ ]:
%%file consumer_velocity.py
from kafka import KafkaConsumer
from collections import defaultdict
from datetime import datetime
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='velocity-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# {user_id: [datetime, datetime, ...]}
user_timestamps = defaultdict(list)
WINDOW_SECONDS = 60
ALERT_THRESHOLD = 3

print("Monitoring velocity anomalies (>3 transactions / 60s per user)...")

for msg in consumer:
    tx = msg.value
    user_id = tx['user_id']
    now = datetime.fromisoformat(tx['timestamp'])

    # Keep only timestamps within the 60-second window
    user_timestamps[user_id] = [
        ts for ts in user_timestamps[user_id]
        if (now - ts).total_seconds() <= WINDOW_SECONDS
    ]
    user_timestamps[user_id].append(now)

    count = len(user_timestamps[user_id])
    if count > ALERT_THRESHOLD:
        print(
            f"VELOCITY ALERT: {user_id} made {count} transactions in the last "
            f"{WINDOW_SECONDS}s | latest: {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}"
        )